# Sales Forecasting

## 📊 Business Context
Predict future revenue.

**Analytical Approach:** Forecasting
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Data Generation
def generate_timeseries(n=365):
    np.random.seed(42)
    dates = pd.date_range(start='2022-01-01', periods=n, freq='D')
    t = np.arange(n)
    
    # Components
    trend = 0.5 * t
    season = 20 * np.sin(2 * np.pi * t / 30) + 10 * np.sin(2 * np.pi * t / 7) # Monthly + Weekly
    noise = np.random.normal(0, 5, n)
    events = np.zeros(n)
    events[::60] = 50 # Spikes every 2 months
    
    values = 100 + trend + season + noise + events
    
    return pd.DataFrame({'Date': dates, 'Revenue': values}).set_index('Date')

df = generate_timeseries(1095)
print(f'Time Series Length: {len(df)} days')
df.plot(figsize=(14,6), title='Historical Data', linewidth=1)
plt.show()

In [ ]:
# Exploratory Data Analysis (EDA)
def analyze_timeseries(df, target):
    # 1. Decomposition
    decomposition = seasonal_decompose(df[target], model='additive', period=30)
    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    plt.show()
    
    # 2. Stationarity Test (ADF)
    result = adfuller(df[target])
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    if result[1] < 0.05:
        print('Result: Series is Stationary')
    else:
        print('Result: Series is Non-Stationary (Differencing may be needed)')
        
    # 3. ACF & PACF
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(df[target], ax=ax1, lags=40)
    plot_pacf(df[target], ax=ax2, lags=40)
    plt.show()

analyze_timeseries(df, 'Revenue')

In [ ]:
# Forecasting Engine
class Forecaster:
    def __init__(self, df, target):
        self.df = df
        self.target = target
        self.models = {}
        
    def train_evaluate(self, test_days=30):
        # Split
        train = self.df.iloc[:-test_days]
        test = self.df.iloc[-test_days:]
        
        # Model 1: Holt-Winters (Triple Exponential Smoothing)
        hw_model = ExponentialSmoothing(
            train[self.target], 
            seasonal_periods=30, 
            trend='add', 
            seasonal='add'
        ).fit()
        self.models['Holt-Winters'] = hw_model.forecast(test_days)
        
        # Model 2: Simple Moving Average (Baseline)
        self.models['Moving Average (7D)'] = test.copy()
        self.models['Moving Average (7D)'][self.target] = train[self.target].rolling(window=7).mean().iloc[-1]
        self.models['Moving Average (7D)'] = self.models['Moving Average (7D)'][self.target]
        
        # Evaluation
        results = []
        plt.figure(figsize=(14, 7))
        plt.plot(train.index[-60:], train[self.target][-60:], label='Train (Last 60 Days)')
        plt.plot(test.index, test[self.target], label='Actual Test', color='black', linewidth=2)
        
        for name, preds in self.models.items():
            mae = mean_absolute_error(test[self.target], preds)
            rmse = np.sqrt(mean_squared_error(test[self.target], preds))
            results.append({'Model': name, 'MAE': mae, 'RMSE': rmse})
            
            plt.plot(test.index, preds, label=f'{name} (MAE={mae:.1f})', linestyle='--')
            
        plt.title('Forecast Model Comparison')
        plt.legend()
        plt.show()
        
        return pd.DataFrame(results)

forecaster = Forecaster(df, 'Revenue')
metrics = forecaster.train_evaluate(test_days=30)
display(metrics)

In [ ]:
# Future Forecast
best_model_name = metrics.sort_values('MAE').iloc[0]['Model']
print(f'Generating future forecast using best model: {best_model_name}')

# Refit on full data
final_model = ExponentialSmoothing(
    df['Revenue'], 
    seasonal_periods=30, 
    trend='add', 
    seasonal='add'
).fit()

future_days = 60
future_forecast = final_model.forecast(future_days)

plt.figure(figsize=(14, 6))
plt.plot(df.index[-90:], df['Revenue'][-90:], label='Historical (Last 90 Days)')
plt.plot(future_forecast.index, future_forecast, label='Future Forecast', color='green', linestyle='--')
plt.title(f'Future {future_days}-Day Forecast')
plt.legend()
plt.show()

## 📈 Strategic Insights

1. **Trend Analysis**: The overall trend indicates a steady increase/decrease, suggesting...
2. **Seasonality**: Strong monthly patterns observed. Peak demand occurs around...
3. **Anomalies**: Spikes in the data correlate with specific events, requiring buffer stock planning.
4. **Action Plan**: Optimize inventory levels based on the 30-day forecast to reduce holding costs.